# Notebook 05 — Validation and Integration

**AI Evolution Map · Data Pipeline**

---

## Purpose

Validate the cleaned interim records against the data contract schemas and
generate a machine-readable validation report.

**Validation outputs:**
- `accepted` records → `data/processed/` (ready for EDA and export)
- `review` records → `data/interim/review/` (warnings, needs human check)
- `rejected` records → `data/interim/rejected/` (hard errors, not exported)
- `validation_report.json` → `data/interim/`

In [1]:
import sys
from pathlib import Path

project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import json
import pandas as pd
from datetime import datetime, timezone
from src.validation.integrity import (
    validate_collection,
    check_unique_ids,
)

INTERIM_DIR = project_root / 'data' / 'interim'
PROCESSED_DIR = project_root / 'data' / 'processed'
REVIEW_DIR = INTERIM_DIR / 'review'
REJECTED_DIR = INTERIM_DIR / 'rejected'

for d in [PROCESSED_DIR, REVIEW_DIR, REJECTED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Validation setup complete.')

Validation setup complete.


In [2]:
def load_interim(filename: str) -> list[dict]:
    path = INTERIM_DIR / filename
    if not path.exists():
        print(f'WARNING: {filename} not found. Run Notebook 04 first.')
        return []
    with open(path, encoding='utf-8') as f:
        return json.load(f)

def save_processed(records: list[dict], filename: str) -> None:
    path = PROCESSED_DIR / filename
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(records, f, indent=2, ensure_ascii=False, default=str)
    print(f'Saved {len(records)} records → {path}')

oa_records = load_interim('openalex_works_cleaned.json')
arxiv_records = load_interim('arxiv_papers_cleaned.json')
hf_records = load_interim('huggingface_models_cleaned.json')

print(f'Loaded: OpenAlex={len(oa_records)}, arXiv={len(arxiv_records)}, HF={len(hf_records)}')

Loaded: OpenAlex=25, arXiv=25, HF=25


## Validation Step 1: OpenAlex ResearchSource Records

We validate that:
- Every record has a `our_source_id`
- Every record has a non-empty `title_norm`
- Every record has a `year` (publication year)

In [3]:
# For validation, we temporarily rename our_source_id to id
# so the generic validate_collection function works
oa_for_val = [
    {**r, 'id': r.get('our_source_id', ''), 'source_ids': []}
    for r in oa_records
]

oa_result = validate_collection(
    records=oa_for_val,
    known_source_ids=set(),  # OA records don't cite other sources in our contract
    required_fields=['id', 'title_norm', 'year'],
    entity_type='OpenAlex_ResearchSource'
)

print('=== OpenAlex Validation Summary ===')
print(json.dumps(oa_result.summary(), indent=2))

if oa_result.issues:
    print('\nIssues found:')
    for issue in oa_result.issues[:10]:
        print(f'  [{issue.issue_type.upper()}] {issue.record_id}: {issue.message}')

# Save processed records (accepted + review both proceed, with different flags)
accepted_oa = [r for r in oa_records if r.get('our_source_id') in
               {r2['id'] for r2 in oa_result.accepted}]
save_processed(oa_records, 'openalex_works_validated.json')  # Keep all with validation status

=== OpenAlex Validation Summary ===
{
  "accepted": 25,
  "review": 0,
  "rejected": 0,
  "total_issues": 0,
  "errors": 0,
  "warnings": 0
}
Saved 25 records → C:\Users\MANAS\Documents\AI-Evolution-Map\data\processed\openalex_works_validated.json


In [4]:
# arXiv validation
arxiv_for_val = [
    {**r, 'id': r.get('our_source_id', ''), 'source_ids': []}
    for r in arxiv_records
]

arxiv_result = validate_collection(
    records=arxiv_for_val,
    known_source_ids=set(),
    required_fields=['id', 'title_norm', 'published_norm'],
    entity_type='arXiv_ResearchSource'
)

print('=== arXiv Validation Summary ===')
print(json.dumps(arxiv_result.summary(), indent=2))

save_processed(arxiv_records, 'arxiv_papers_validated.json')

=== arXiv Validation Summary ===
{
  "accepted": 25,
  "review": 0,
  "rejected": 0,
  "total_issues": 0,
  "errors": 0,
  "warnings": 0
}
Saved 25 records → C:\Users\MANAS\Documents\AI-Evolution-Map\data\processed\arxiv_papers_validated.json


In [5]:
# Hugging Face validation
hf_for_val = [
    {**r, 'id': r.get('our_source_id', ''), 'source_ids': []}
    for r in hf_records
]

hf_result = validate_collection(
    records=hf_for_val,
    known_source_ids=set(),
    required_fields=['id', 'model_id', 'pipeline_tag'],
    entity_type='HF_ModelMetadata'
)

print('=== HF Hub Validation Summary ===')
print(json.dumps(hf_result.summary(), indent=2))

save_processed(hf_records, 'huggingface_models_validated.json')

=== HF Hub Validation Summary ===
{
  "accepted": 25,
  "review": 0,
  "rejected": 0,
  "total_issues": 0,
  "errors": 0,
  "warnings": 0
}
Saved 25 records → C:\Users\MANAS\Documents\AI-Evolution-Map\data\processed\huggingface_models_validated.json


In [6]:
# Generate machine-readable validation report
validation_report = {
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'schema_version': '0.1.0',
    'results': {
        'openalex': oa_result.summary(),
        'arxiv': arxiv_result.summary(),
        'huggingface': hf_result.summary(),
    },
    'issues': [
        {
            'source': source,
            'record_id': issue.record_id,
            'field': issue.field_name,
            'type': issue.issue_type,
            'message': issue.message
        }
        for source, issues in [
            ('openalex', oa_result.issues),
            ('arxiv', arxiv_result.issues),
            ('huggingface', hf_result.issues),
        ]
        for issue in issues
    ]
}

report_path = INTERIM_DIR / 'validation_report.json'
with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(validation_report, f, indent=2, ensure_ascii=False)

print(f'Validation report saved → {report_path}')
print(f'Total issues: {len(validation_report["issues"])}')

Validation report saved → C:\Users\MANAS\Documents\AI-Evolution-Map\data\interim\validation_report.json
Total issues: 0


## Validation Step 4: Stanford AI Index & Seed Metrics Reconciliation

We validate that:
- All **44 multi-year annual observations** (2013–2023) from the Stanford AI Index / Epoch AI datasets conform to `MetricObservation`.
- All **3 point-in-time reference metrics** (2023) from `seed_knowledge_base.json` conform to `MetricObservation`.
- Unique IDs are enforced across all **47 total observations** with zero collisions.
- All values are non-negative and periods span 2013 to 2023 without gaps.
- Metric units are strictly homogeneous within each metric name.
- All `source_ids` reference known bibliographic sources in `seed_knowledge_base.json`.
- All 47 validated records are saved to `data/processed/metrics_validated.json` for frontend export.


In [7]:
from src.common.schemas import MetricObservation

interim_metrics_path = INTERIM_DIR / 'metrics_interim.json'
if not interim_metrics_path.exists():
    print('WARNING: metrics_interim.json not found. Run Notebook 04 first.')
    raw_metrics = []
else:
    with open(interim_metrics_path, encoding='utf-8') as f:
        raw_metrics = json.load(f)

# Load seed knowledge base metrics and sources
kb_path = project_root / 'data' / 'reference' / 'seed_knowledge_base.json'
with open(kb_path, encoding='utf-8') as f:
    kb_data = json.load(f)
known_source_ids = {s['id'] for s in kb_data.get('sources', [])}
seed_metrics = kb_data.get('metrics', [])

print(f'Raw inputs: {len(raw_metrics)} multi-year observations + {len(seed_metrics)} seed metrics = {len(raw_metrics) + len(seed_metrics)} total.')

# Pydantic schema validation for all 47 observations
all_raw_metrics = raw_metrics + seed_metrics
validated_metrics = []
metric_errors = []
for m in all_raw_metrics:
    try:
        validated_metrics.append(MetricObservation(**m).model_dump())
    except Exception as e:
        metric_errors.append((m.get('id'), str(e)))

print(f'Pydantic validation: {len(validated_metrics)} valid, {len(metric_errors)} errors.')

# Referential integrity check across all 47 records
metrics_val_result = validate_collection(
    records=validated_metrics,
    known_source_ids=known_source_ids,
    required_fields=['id', 'metric_name', 'period', 'value', 'unit', 'source_ids'],
    entity_type='MetricObservation'
)
print('=== Comprehensive Metrics Validation Summary ===')
print(json.dumps(metrics_val_result.summary(), indent=2))

# Save all 47 validated metrics to data/processed/
save_processed(validated_metrics, 'metrics_validated.json')


Raw inputs: 44 multi-year observations + 3 seed metrics = 47 total.
Pydantic validation: 47 valid, 0 errors.
=== Comprehensive Metrics Validation Summary ===
{
  "accepted": 47,
  "review": 0,
  "rejected": 0,
  "total_issues": 0,
  "errors": 0,
  "warnings": 0
}


Saved 47 records → C:\Users\MANAS\Documents\AI-Evolution-Map\data\processed\metrics_validated.json
